In [ ]:
from dotenv import load_dotenv
import os
from datetime import datetime
from openai import OpenAI
import json

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

client = OpenAI(api_key=OPENAI_API_KEY)

def calculate_age(birthdate):
    today_year = datetime.today().year
    birth_obj = datetime.strptime(birthdate,"%Y-%m-%d")
    birth_year = birth_obj.year
    
    return today_year - birth_year


def convert_currency(amount, from_currency, to_currency, rate):
    return amount*rate

def calculate_bmi(height, weight):
    return weight/(height/100)**2


tools = [
    {
        "type": "function",
        "name": "calculate_age",
        "description": "생년월일을 입력받아 현재 년도 기준의 나이를 계산합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "birthdate": {
                    "type": "string",
                    "description": "생년월일 (형식: YYYY-MM-DD)"
                }
            },
            "required": ["birthdate"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "convert_currency",
        "description": "지정된 환율을 적용하여 금액을 다른 통화로 변환합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {
                    "type": "number",
                    "description": "변환할 금액"
                },
                "from_currency": {
                    "type": "string",
                    "description": "기존 통화 코드 (예: USD)"
                },
                "to_currency": {
                    "type": "string",
                    "description": "변환할 통화 코드 (예: KRW)"
                },
                "rate": {
                    "type": "number",
                    "description": "적용할 환율 값"
                }
            },
            "required": ["amount", "from_currency", "to_currency", "rate"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "calculate_bmi",
        "description": "키(cm)와 몸무게(kg)를 기반으로 체질량지수(BMI)를 계산합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "height": {
                    "type": "number",
                    "description": "키 (단위: cm)"
                },
                "weight": {
                    "type": "number",
                    "description": "몸무게 (단위: kg)"
                }
            },
            "required": ["height", "weight"],
            "additionalProperties": False
        },
        "strict": True
    }
]

input_messages = []

while True:
    user_input = input("\n사용자: ")
    if user_input == "종료":
        break
    
    input_messages.append({
        "role": "user",
        "content": user_input
    })

    response = client.responses.create(
        model="gpt-5.5",  
        input=input_messages,
        tools=tools
    )
    # print(response.output)
    # print(response.output[0].type)
    # [ResponseFunctionToolCall(arguments='{"birthdate":"2000-12-18"}', call_id='call_LCXUl91jVKCoyCePwbMgSMEX', name='calculate_age', type='function_call', id='fc_05799e89cbb1992b006a4f31508b348198a124ecb16efed653', namespace=None, status='completed')]
    # function_call
    if response.output and response.output[0].type == "function_call":
        tool_call = response.output[0]
        # print(tool_call)

        input_messages.append(tool_call)

        args = json.loads(tool_call.arguments)
        result = None
        # print("args : ",args)
        # args :  {'birthdate': '2000-12-18'}
        
        if tool_call.name == "calculate_age":
            result = calculate_age(**args)
        elif tool_call.name == "convert_currency":
            result = convert_currency(**args)
        elif tool_call.name == "calculate_bmi":
            result = calculate_bmi(**args)

        # print("result : ", result, type(result))
        # result :  26 <class 'int'>

        input_messages.append({
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": str(result)
        })

        # print("현재 input_messages : ",input_messages)

        final_response = client.responses.create(
            model="gpt-5.5",
            input=input_messages,
            tools=tools
        )

        # print("final_response : ",final_response)

        print(f"AI : {final_response.output_text}")
        
        # 처음 안 부분인데 이렇게 하면 전에 질문했던 것도 기억한다고 한다
        input_messages.append(final_response.output[0])
        
    else:
        print(f"AI : {response.output_text}")
        # 처음 안 부분
        input_messages.append(response.output[0])

AI : 입력하신 `-1-12-01`은 생년월일 형식이 불명확합니다.

혹시 `2001-12-01`생을 뜻하신 거라면, 현재 기준 **만 24세**입니다.  
생일인 2026년 12월 1일이 지나면 **만 25세**가 됩니다.
